# RobustBench-LLM: QLoRA Classifier
Fine-tuning Llama 3.2 1B for Security Classification

In [ ]:
!pip install -q torch transformers==4.46.1 datasets==3.0.1 peft==0.13.0 bitsandbytes==0.44.1 trl==0.12.0 accelerate==0.34.2 huggingface_hub==0.26.2

In [ ]:
import torch
from huggingface_hub import login

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print("\n--- You must login with your HuggingFace Token to download Llama ---")
login()

In [ ]:
import os
from google.colab import files

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
MODEL_TYPE = 'causal'

if not os.path.exists("train.jsonl"):
    print("Upload your 3 JSONL files from data/processed/ (train, val, test):")
    uploaded = files.upload()
else:
    print("Files already uploaded!")

In [ ]:
import json
from datasets import Dataset

def load_and_format(path):
    samples = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            text = f"System: You are a security classifier. Answer with just 'adversarial' or 'benign'.\nUser: {item['prompt'][:800]}\nAssistant: {item['label']}"
            samples.append({'text': text})
    return Dataset.from_list(samples)

train_dataset = load_and_format('train.jsonl')
val_dataset = load_and_format('val.jsonl')
print(f"Data mapped! Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type='nf4', 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map='auto'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("Model cleanly loaded using 4-bit precision!")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules='all-linear',
    lora_dropout=0.05, 
    bias='none',
    task_type='CAUSAL_LM',
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

print(f"Starting QLoRA training...")
start = time.time()

training_args = SFTConfig(
    output_dir='./classifier_output',
    max_seq_length=512,
    num_train_epochs=3, 
    per_device_train_batch_size=16, 
    learning_rate=2e-4,
    fp16=True, 
    dataset_text_field='text', 
    report_to='none', 
    save_strategy='epoch', 
    logging_steps=10
)

trainer = SFTTrainer(
    model=model, 
    args=training_args, 
    train_dataset=train_dataset, 
    eval_dataset=val_dataset, 
    processing_class=tokenizer,
       
)

trainer.train()
print(f"\nTraining completed in {(time.time()-start)/60:.1f} minutes!")

In [ ]:
import os
import json
import shutil
from google.colab import files

model.save_pretrained('./classifier_adapter')
tokenizer.save_pretrained('./classifier_adapter')

with open('./classifier_adapter/robustbench_config.json', 'w') as f:
    json.dump({'model_id': MODEL_ID, 'model_type': MODEL_TYPE}, f)

shutil.make_archive('classifier_adapter', 'zip', './classifier_adapter')
print("Downloading... place the zip contents in models/classifier/final_adapter/ on your Mac!")
files.download('classifier_adapter.zip')